# 01 - Crawling
Mengambil **100 berita `sport`** dan **100 berita `finance`** langsung dari detik.com.

Alur: halaman indeks → kumpulkan link artikel → buka tiap artikel → ambil judul + isi → simpan ke `dataset_detik.csv`.

In [ ]:
import re, time
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0"}
TARGET = 100
SITUS = {"sport": "sport.detik.com", "finance": "finance.detik.com"}

def buka(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

### 1. Kumpulkan link artikel dari halaman indeks
Link artikel detik selalu mengandung pola `/d-<angka>`.

In [ ]:
def ambil_link(domain):
    link = []

    for page in range(1, 21):
        for a in buka(f"https://{domain}/indeks?page={page}").select("a[href]"):
            url = a["href"].split("?")[0]

            if url.startswith(f"https://{domain}") and re.search(r"/d-\d+", url) and url not in link:
                link.append(url)

        if len(link) >= TARGET * 1.5:
            break

        time.sleep(1)

    return link

### 2. Ambil judul + isi satu artikel
Elemen tidak relevan (script, iklan, gambar) dibuang, lalu spasi dirapikan.

In [ ]:
def ambil_isi(url):
    soup = buka(url)
    judul = soup.select_one("h1")
    isi = soup.select_one(".detail__body-text, .detail__body, .itp_bodycontent, article")
    if isi is None:
        return ""
    for x in isi.select("script, style, aside, iframe, figure, .detail__media, .parallax, .ad-container"):
        x.decompose()
    teks = " ".join(isi.get_text(" ", strip=True).split())
    judul = judul.get_text(" ", strip=True) if judul else ""
    return f"{judul}. {teks}" if judul and judul.lower() not in teks.lower() else teks

### 3. Crawling semua kategori
Artikel dengan isi < 200 karakter dilewati. ID: `SP001`-`SP100` dan `FN001`-`FN100`.

In [ ]:
baris = []
for label, domain in SITUS.items():
    n = 0
    for url in ambil_link(domain):
        try:
            teks = ambil_isi(url)
        except requests.RequestException:
            continue
        if len(teks) < 200:
            continue
        n += 1
        baris.append({"id": ("SP" if label == "sport" else "FN") + f"{n:03d}",
                      "isi_berita": teks, "label": label, "url": url})
        time.sleep(1)
        if n == TARGET:
            break
    print(label, n, "artikel")

df = pd.DataFrame(baris)
df.to_csv("dataset_detik.csv", index=False, encoding="utf-8-sig")
df.head()

In [ ]:
print(df["label"].value_counts())
print("URL duplikat:", df["url"].duplicated().sum())

sport 100 artikel
finance 100 artikel


,id,isi_berita,label,url
0,SP001,Profil Desak Made Peraih Emas Keempat RI: Kena...,sport,https://sport.detik.com/sport-lain/d-8686722/p...
1,SP002,"Gacor di Asian Games, Leo/Daniel Kini Bidik Tu...",sport,https://sport.detik.com/raket/d-8686185/gacor-...
2,SP003,Asian Games 2026: Desak Made Raih Emas Keempat...,sport,https://sport.detik.com/sport-lain/d-8686648/a...
3,SP004,"Panjat Tebing Buru Medali, Yenni Wahid: Aditya...",sport,https://sport.detik.com/sport-lain/d-8686152/p...
4,SP005,Mau Nonton FIA Rallycross World Cup Indonesia?...,sport,https://sport.detik.com/sportstyle/d-8686158/m...
